# Adapter Analysis — Branch I (no S matrix)
Runs the full analysis pipeline from saved weights for the older model variant that uses a direct weight matrix `W` instead of `(g0, S)`.

**Inputs (all loaded from disk):**
- Weights: `W`, `om`
- Ratemaps: pre-computed `.npy` files (or recomputed from weights)
- Losses + lambdas (optional)
- Positions + representations for manifold (optional)

In [ ]:
import os, sys, pickle, json, logging
import numpy as np
from tqdm import tqdm
import mlflow

# ── Add Pilot Decoder to path so analysis helpers are importable ───────────────
PILOT_DECODER = os.path.abspath('.')
if PILOT_DECODER not in sys.path:
    sys.path.insert(0, PILOT_DECODER)

from analysis import (
    frequency_plot,
    s_matrix_plot,
    loss_plots,
    quantitative_analysis,
    neuron_plotter_2d,
    module_ratemaps_plot,
    grid_type,
    manifold,
    log_figure,
)
from scores import GridScorer

# ── Configuration ──────────────────────────────────────────────────────────────
FILEPATH = '/home/julian/Projects/MasterThesis/ICLR_Actionable_Reps/data/251106/170735'
COUNTER  = 0           # which saved checkpoint

GENERATE_MANIFOLD = True

sep = '_' if COUNTER != '' else ''
print(f'Run directory : {FILEPATH}')
print(f'Counter       : {COUNTER}')

## 1 · Load weights

In [ ]:
def _load_pkl(path):
    with open(path, 'rb') as f:
        obj = pickle.load(f)
    return np.array(obj)   # convert JAX arrays to numpy


def _pkl(name):
    return os.path.join(FILEPATH, f'{name}{sep}{COUNTER}.pkl')


# Prefer final weights if available
W  = _load_pkl(_pkl('W_final') if os.path.exists(_pkl('W_final')) else _pkl('W'))
om = _load_pkl(_pkl('om_final') if os.path.exists(_pkl('om_final')) else _pkl('om'))

D = W.shape[0]
M = om.shape[0]
print(f'D={D},  M={M}')
print(f'W : {W.shape},  om : {om.shape}')

## 2 · Load / compute ratemaps
Pre-saved `.npy` files are used when available; otherwise ratemaps are recomputed from `W` and `om` using a pure-NumPy implementation of `init_irreps_2D`.

In [ ]:
def init_irreps_2D_np(om: np.ndarray, phi: np.ndarray) -> np.ndarray:
    """NumPy port of ICLR `init_irreps_2D`.

    Args:
        om  : (M, 2) frequencies
        phi : (N, 2) positions

    Returns:
        I   : (2*M+1, N)  — [1, cos(k_1·phi), sin(k_1·phi), …, cos(k_M·phi), sin(k_M·phi)]
    """
    N = phi.shape[0]
    M_freq = om.shape[0]
    # k · phi  →  (M, N)
    k_dot_phi = (om @ phi.T)          # (M, N)
    I = np.ones((2 * M_freq + 1, N))
    I[1::2, :] = np.cos(k_dot_phi)   # rows 1, 3, 5, …
    I[2::2, :] = np.sin(k_dot_phi)   # rows 2, 4, 6, …
    return I


def get_ratemaps_np(W, om, res, widths):
    """Compute ratemaps, returning (D, res*res) arrays (flattened)."""
    maps = []
    for w in widths:
        xs  = np.linspace(-w / 2, w / 2, res)
        gx, gy = np.meshgrid(xs, xs)
        phi = np.stack([gx.ravel(), gy.ravel()], axis=1)   # (res^2, 2)
        I   = init_irreps_2D_np(om, phi)                   # (2*M+1, res^2)
        V   = W @ I                                         # (D, res^2)
        maps.append(V)
    return maps


res    = 70
widths = (1, 2, 4)
scale_tags = ('small', 'medium', 'large')

Vs = []
for tag, w in zip(scale_tags, widths):
    npy_path = os.path.join(FILEPATH, f'ratemaps_{tag}_{COUNTER}.npy')
    if os.path.exists(npy_path):
        V = np.load(npy_path)                    # (D, res, res)
        V = V.reshape(D, res * res)              # → (D, res*res)
        print(f'Loaded {tag} ratemaps from {npy_path}')
    else:
        print(f'Computing {tag} ratemaps (w={w})…')
        V = get_ratemaps_np(W, om, res, [w])[0]
    Vs.append(V)

V_small, V_medium, V_large = Vs
print(f'Ratemap shapes — small:{V_small.shape}  medium:{V_medium.shape}  large:{V_large.shape}')

## 3 · Load losses and lambdas (optional)

In [ ]:
# L has shape (4, T): rows are [total_loss, separation, positivity, norm]
L_path = _pkl('L')
L = _load_pkl(L_path) if os.path.exists(L_path) else None

lambda_pos_path  = _pkl('lambda_pos')
lambda_norm_path = _pkl('lambda_norm')
lambda_pos  = _load_pkl(lambda_pos_path)  if os.path.exists(lambda_pos_path)  else None
lambda_norm = _load_pkl(lambda_norm_path) if os.path.exists(lambda_norm_path) else None

if L is not None:
    print(f'L shape: {L.shape}  (rows: total, separation, positivity, norm)')
    train_losses = {
        'loss'       : L[0],
        'separation' : L[1],
        'positivity' : L[2],
        'norm'       : L[3],
    }
else:
    print('No loss file found — loss plots will be skipped.')
    train_losses = None

if lambda_pos is not None:
    print(f'lambda_pos shape: {lambda_pos.shape}')
if lambda_norm is not None:
    print(f'lambda_norm shape: {lambda_norm.shape}')

## 4 · Load manifold data (optional)

In [ ]:
def _npy(name):
    return os.path.join(FILEPATH, f'{name}_{COUNTER}.npy')


positions_path       = _npy('positions_manifold')
representations_path = _npy('representations')

positions       = np.load(positions_path)       if os.path.exists(positions_path)       else None
representations = np.load(representations_path) if os.path.exists(representations_path) else None

if positions is not None and representations is not None:
    print(f'Positions       : {positions.shape}')        # (B, L, 2)
    print(f'Representations : {representations.shape}')  # (B, L, D)
else:
    print('Manifold data not found — manifold analysis will be skipped.')
    GENERATE_MANIFOLD = False

In [ ]:
# ── MLflow experiment + run ────────────────────────────────────────────────────
_sweep = os.path.basename(os.path.dirname(FILEPATH))   # e.g. 251106
_run   = os.path.basename(FILEPATH)                    # e.g. 170735
RUN_NAME = f'{_sweep}/{_run}/k{COUNTER}'

mlflow.set_tracking_uri(f"sqlite:///{PILOT_DECODER}/mlruns.db")
mlflow.set_experiment('adapter_analysis')

active_run = mlflow.start_run(run_name=RUN_NAME)
print(f'MLflow experiment : adapter_analysis')
print(f'Run name          : {RUN_NAME}')
print(f'Run ID            : {active_run.info.run_id}')

# Log provenance parameters
params_path = os.path.join(FILEPATH, 'parameters.json')
if os.path.exists(params_path):
    with open(params_path) as f:
        params = json.load(f)
    mlflow.log_params({k: v for k, v in params.items() if isinstance(v, (int, float, str, bool))})
mlflow.log_param('filepath', FILEPATH)
mlflow.log_param('counter', COUNTER)

---
## Analysis

### Frequency vectors

In [ ]:
fig_freq = frequency_plot(om)
log_figure(fig_freq, f'freq_plot_k{COUNTER}')
fig_freq.show()

### W matrix

In [ ]:
import plotly.graph_objects as go

fig_W = go.Figure(go.Heatmap(z=W, colorscale='RdBu_r', zmid=0))
fig_W.update_layout(
    title='W matrix',
    xaxis_title='Irrep index',
    yaxis_title='Neuron index',
    xaxis=dict(constrain='domain'),
    yaxis=dict(scaleanchor='x', scaleratio=1, constrain='domain'),
    height=600, width=600,
)
log_figure(fig_W, f'W_matrix_k{COUNTER}')
fig_W.show()

### Loss curves

In [ ]:
if train_losses is not None:
    fig_loss = loss_plots(train_losses, lambda_pos=lambda_pos, lambda_norm=lambda_norm)
    log_figure(fig_loss, f'loss_curves_k{COUNTER}')
    fig_loss.show()
else:
    print('Skipped — no loss data.')

### Grid scores

In [ ]:
fig_scores, scores = quantitative_analysis(Vs, widths, res)
log_figure(fig_scores, f'grid_scores_k{COUNTER}')
fig_scores.show()

# Log scalar metrics
for key, value in scores.items():
    if isinstance(value, (int, float)):
        mlflow.log_metric(f'k{COUNTER}/{key}', value)

print(f"\nMax  grid scores — sm:{scores['sm_60_max']:.3f}  md:{scores['md_60_max']:.3f}  lg:{scores['lg_60_max']:.3f}")
print(f"Mean grid scores — sm:{scores['sm_60_mean']:.3f}  md:{scores['md_60_mean']:.3f}  lg:{scores['lg_60_mean']:.3f}")

### Neuron ratemaps

In [ ]:
for V, tag, score_key in [
    (V_small,  'small',  'sm_60'),
    (V_medium, 'medium', 'md_60'),
    (V_large,  'large',  'lg_60'),
]:
    fig = neuron_plotter_2d(V, res, scores[score_key])
    log_figure(fig, f'neurons_{tag}_k{COUNTER}_scores_60')
    fig.show()

for V, tag, score_key in [
    (V_small,  'small',  'sm_90'),
    (V_medium, 'medium', 'md_90'),
    (V_large,  'large',  'lg_90'),
]:
    fig = neuron_plotter_2d(V, res, scores[score_key])
    log_figure(fig, f'neurons_{tag}_k{COUNTER}_scores_90')
    fig.show()

### Module extraction and ratemaps

In [ ]:
large_width  = widths[2]
coord_range  = ((-large_width / 2, large_width / 2),) * 2
starts       = [0.2] * 10
ends         = np.linspace(0.4, 1.0, num=10)
scorer       = GridScorer(res, coord_range, zip(starts, ends.tolist()))

sacs   = [scorer.calculate_sac(V_large[i].reshape(res, res)) for i in range(D)]
labels, _ = scorer.get_modules(sacs, max_m=15)
labels = np.array(labels)
n_modules = int(labels.max()) + 1

print(f'{n_modules} modules found')
for m in range(n_modules):
    print(f'  Module {m}: {(labels == m).sum()} neurons')

In [ ]:
fig_mod = module_ratemaps_plot(V_large, res, scores['lg_60'], labels)
log_figure(fig_mod, f'module_ratemaps_k{COUNTER}')
fig_mod.show()

In [ ]:
fig_gt = grid_type(scores['lg_60'], scores['lg_90'], labels)
log_figure(fig_gt, f'grid_type_k{COUNTER}')
fig_gt.show()

### Manifold analysis

In [ ]:
if not GENERATE_MANIFOLD:
    print('Manifold analysis skipped (data not available).')
else:
    import tempfile
    B, L_len, _ = representations.shape
    reps_flat   = representations.reshape(B * L_len, D)   # (B*L, D)

    for module_idx in range(n_modules):
        mask        = labels == module_idx
        module_reps = reps_flat[:, mask]                   # (B*L, Dm)

        result = manifold(positions, module_reps, module_idx)
        if result is None:
            print(f'Module {module_idx}: manifold embedding failed, skipping.')
            continue

        manifold_fig, scree_fig, slice_xy, slice_xz, slice_yz, traj_fig, module_positions, module_embedding = result

        log_figure(scree_fig,    f'scree_module{module_idx}_k{COUNTER}')
        log_figure(manifold_fig, f'manifold_module{module_idx}_k{COUNTER}')
        log_figure(slice_xy,     f'manifold_slice_xy_module{module_idx}_k{COUNTER}')
        log_figure(slice_xz,     f'manifold_slice_xz_module{module_idx}_k{COUNTER}')
        log_figure(slice_yz,     f'manifold_slice_yz_module{module_idx}_k{COUNTER}')
        mlflow.log_figure(traj_fig, f'figures/trajectory_module{module_idx}_k{COUNTER}.png')

        with tempfile.TemporaryDirectory() as tmpdir:
            pos_path = f'{tmpdir}/positions_module{module_idx}.npy'
            emb_path = f'{tmpdir}/embedding_module{module_idx}.npy'
            np.save(pos_path, module_positions)
            np.save(emb_path, module_embedding)
            mlflow.log_artifact(pos_path, artifact_path=f'manifold/k{COUNTER}')
            mlflow.log_artifact(emb_path, artifact_path=f'manifold/k{COUNTER}')

        print(f'── Module {module_idx} ──────────────────────────────────')
        scree_fig.show()
        manifold_fig.show()
        slice_xy.show()
        slice_xz.show()
        slice_yz.show()

In [ ]:
mlflow.end_run()
print(f'Run {active_run.info.run_id} finished.')